# Wine Classification with a PyTorch Neural Network

This notebook converts the assignment into a step-by-step workflow for:
- loading the Wine dataset
- preprocessing and scaling the features
- building a neural network in PyTorch
- training and evaluating the model
- experimenting with different architectures and hyperparameters

Note: in scikit-learn, the wine labels are encoded as `0`, `1`, and `2`.

## 1. Imports and Reproducibility

Import the required libraries and set random seeds so results are more reproducible.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

print("Seeds set for reproducibility.")
print("PyTorch version:", torch.__version__)

Seeds set for reproducibility.
PyTorch version: 2.1.2+cu121


## 2. Load and Preview the Wine Dataset

Load the dataset, inspect the feature names and class names, and preview the shape and a few sample rows.

In [2]:
wine = load_wine()
X, y = wine.data, wine.target

print("Feature names:")
print(wine.feature_names)

print("\nClass names:")
print(wine.target_names)

print("\nDataset shape:", X.shape)

print("\nFirst 5 samples:")
print(X[:5])

print("\nFirst 5 labels:")
print(y[:5])

Feature names:
['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']

Class names:
['class_0' 'class_1' 'class_2']

Dataset shape: (178, 13)

First 5 samples:
[[1.423e+01 1.710e+00 2.430e+00 1.560e+01 1.270e+02 2.800e+00 3.060e+00
  2.800e-01 2.290e+00 5.640e+00 1.040e+00 3.920e+00 1.065e+03]
 [1.320e+01 1.780e+00 2.140e+00 1.120e+01 1.000e+02 2.650e+00 2.760e+00
  2.600e-01 1.280e+00 4.380e+00 1.050e+00 3.400e+00 1.050e+03]
 [1.316e+01 2.360e+00 2.670e+00 1.860e+01 1.010e+02 2.800e+00 3.240e+00
  3.000e-01 2.810e+00 5.680e+00 1.030e+00 3.170e+00 1.185e+03]
 [1.437e+01 1.950e+00 2.500e+00 1.680e+01 1.130e+02 3.850e+00 3.490e+00
  2.400e-01 2.180e+00 7.800e+00 8.600e-01 3.450e+00 1.480e+03]
 [1.324e+01 2.590e+00 2.870e+00 2.100e+01 1.180e+02 2.800e+00 2.690e+00
  3.900e-01 1.820e+00 4.320e+00 1.040e+00 2.930e+00 7.350e+02]]

First 5 

## 3. Train/Test Split and Feature Scaling

Split the data into training and testing sets, then standardize the features using only the training data.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape:", X_test_scaled.shape)

print("\nFirst standardized training row:")
print(X_train_scaled[0])

Training set shape: (142, 13)
Test set shape: (36, 13)

First standardized training row:
[ 0.38580089 -0.63787118  1.77666817 -1.22453161  0.69643032  0.52686525
  0.73229212 -0.1695489  -0.41578344 -0.16746725  0.62437819  0.2529082
  0.46772474]


## 4. Convert Data to PyTorch Tensors

Convert the NumPy arrays into tensors. Features use `torch.float32` and labels use `torch.long`.

In [4]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("X_train_tensor:", X_train_tensor.shape, X_train_tensor.dtype)
print("X_test_tensor:", X_test_tensor.shape, X_test_tensor.dtype)
print("y_train_tensor:", y_train_tensor.shape, y_train_tensor.dtype)
print("y_test_tensor:", y_test_tensor.shape, y_test_tensor.dtype)

X_train_tensor: torch.Size([142, 13]) torch.float32
X_test_tensor: torch.Size([36, 13]) torch.float32
y_train_tensor: torch.Size([142]) torch.int64
y_test_tensor: torch.Size([36]) torch.int64


## 5. Define the Neural Network

Create a feed-forward neural network with the architecture `13 → 32 → 16 → 3`.  
The final layer returns raw logits for multi-class classification.

In [5]:
class WineClassifier(nn.Module):
    def __init__(self, input_dim=13, hidden1=32, hidden2=16, output_dim=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.model(x)

## 6. Configure Loss and Optimizer

Initialize the model, use cross-entropy loss for classification, and optimize with Adam.

In [6]:
model = WineClassifier(
    input_dim=X_train_tensor.shape[1],
    hidden1=32,
    hidden2=16,
    output_dim=len(np.unique(y))
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print("\nLoss function:", criterion)
print("Optimizer:", optimizer)

WineClassifier(
  (model): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=3, bias=True)
  )
)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## 7. Train the Model

Run the training loop for multiple epochs and print the loss and training accuracy every few epochs.

In [7]:
epochs = 200
train_losses = []
train_accuracies = []

for epoch in range(epochs):
    model.train()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        predicted_classes = torch.argmax(outputs, dim=1)
        train_accuracy = (predicted_classes == y_train_tensor).float().mean().item()

    train_losses.append(loss.item())
    train_accuracies.append(train_accuracy)

    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1:3d}/{epochs} | "
            f"Loss: {loss.item():.4f} | "
            f"Training Accuracy: {train_accuracy * 100:.2f}%"
        )

Epoch   1/200 | Loss: 1.0969 | Training Accuracy: 33.80%
Epoch  20/200 | Loss: 0.9527 | Training Accuracy: 62.68%
Epoch  40/200 | Loss: 0.7433 | Training Accuracy: 85.21%
Epoch  60/200 | Loss: 0.5047 | Training Accuracy: 90.85%
Epoch  80/200 | Loss: 0.2938 | Training Accuracy: 97.89%
Epoch 100/200 | Loss: 0.1541 | Training Accuracy: 99.30%
Epoch 120/200 | Loss: 0.0854 | Training Accuracy: 99.30%
Epoch 140/200 | Loss: 0.0529 | Training Accuracy: 100.00%
Epoch 160/200 | Loss: 0.0358 | Training Accuracy: 100.00%
Epoch 180/200 | Loss: 0.0258 | Training Accuracy: 100.00%
Epoch 200/200 | Loss: 0.0194 | Training Accuracy: 100.00%


## 8. Evaluate Test Performance

Switch the model to evaluation mode and measure test accuracy.  
A confusion matrix and classification report are also included for more detail.

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_predictions = torch.argmax(test_logits, dim=1)

test_accuracy = accuracy_score(y_test, test_predictions.numpy())
cm = confusion_matrix(y_test, test_predictions.numpy())
report = classification_report(
    y_test,
    test_predictions.numpy(),
    target_names=wine.target_names
)

print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(report)

Test Accuracy: 97.22%

Confusion Matrix:
[[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]

Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



## 9. Experiment with Architecture and Hyperparameters

Try a few variations such as different hidden sizes, dropout, learning rate, epoch count, and optimizer choice.

In [9]:
class FlexibleWineClassifier(nn.Module):
    def __init__(self, input_dim=13, hidden_sizes=(32, 16), output_dim=3, dropout=0.0):
        super().__init__()
        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


def run_experiment(hidden_sizes=(32, 16), lr=0.001, epochs=200, dropout=0.0, optimizer_name="adam"):
    torch.manual_seed(42)

    experiment_model = FlexibleWineClassifier(
        input_dim=X_train_tensor.shape[1],
        hidden_sizes=hidden_sizes,
        output_dim=len(np.unique(y)),
        dropout=dropout
    )

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "adam":
        experiment_optimizer = optim.Adam(experiment_model.parameters(), lr=lr)
    else:
        experiment_optimizer = optim.SGD(experiment_model.parameters(), lr=lr, momentum=0.9)

    for _ in range(epochs):
        experiment_model.train()
        logits = experiment_model(X_train_tensor)
        loss = criterion(logits, y_train_tensor)

        experiment_optimizer.zero_grad()
        loss.backward()
        experiment_optimizer.step()

    experiment_model.eval()
    with torch.no_grad():
        preds = torch.argmax(experiment_model(X_test_tensor), dim=1)
        accuracy = (preds == y_test_tensor).float().mean().item()

    return accuracy

In [10]:
experiment_settings = [
    {"hidden_sizes": (32, 16), "lr": 0.001, "epochs": 200, "dropout": 0.0, "optimizer_name": "adam"},
    {"hidden_sizes": (64, 32), "lr": 0.001, "epochs": 300, "dropout": 0.0, "optimizer_name": "adam"},
    {"hidden_sizes": (64, 32), "lr": 0.0005, "epochs": 400, "dropout": 0.2, "optimizer_name": "adam"},
    {"hidden_sizes": (128, 64), "lr": 0.005, "epochs": 200, "dropout": 0.0, "optimizer_name": "adam"},
    {"hidden_sizes": (32, 16), "lr": 0.01, "epochs": 300, "dropout": 0.0, "optimizer_name": "sgd"},
]

results = []

for config in experiment_settings:
    accuracy = run_experiment(**config)
    result = dict(config)
    result["test_accuracy"] = round(accuracy * 100, 2)
    results.append(result)

sorted_results = sorted(results, key=lambda item: item["test_accuracy"], reverse=True)

print("Experiment Results:")
for index, result in enumerate(sorted_results, start=1):
    print(f"{index}. {result}")

print("\nBest configuration:")
print(sorted_results[0])

Experiment Results:
1. {'hidden_sizes': (32, 16), 'lr': 0.001, 'epochs': 200, 'dropout': 0.0, 'optimizer_name': 'adam', 'test_accuracy': 97.22}
2. {'hidden_sizes': (64, 32), 'lr': 0.001, 'epochs': 300, 'dropout': 0.0, 'optimizer_name': 'adam', 'test_accuracy': 97.22}
3. {'hidden_sizes': (64, 32), 'lr': 0.0005, 'epochs': 400, 'dropout': 0.2, 'optimizer_name': 'adam', 'test_accuracy': 97.22}
4. {'hidden_sizes': (128, 64), 'lr': 0.005, 'epochs': 200, 'dropout': 0.0, 'optimizer_name': 'adam', 'test_accuracy': 97.22}
5. {'hidden_sizes': (32, 16), 'lr': 0.01, 'epochs': 300, 'dropout': 0.0, 'optimizer_name': 'sgd', 'test_accuracy': 97.22}

Best configuration:
{'hidden_sizes': (32, 16), 'lr': 0.001, 'epochs': 200, 'dropout': 0.0, 'optimizer_name': 'adam', 'test_accuracy': 97.22}


## 9a. CNN Experiment on the Same Dataset

Although the Wine dataset is **tabular** rather than image-based, we can still try a small **1D Convolutional Neural Network (CNN)** by reshaping each sample from `(13,)` to `(1, 13)`. This is mainly an experiment to compare CNN behavior with the fully connected network above.

In [11]:
# Reshape tabular features for a 1D CNN: (batch, channels, length)
X_train_cnn = X_train_tensor.unsqueeze(1)
X_test_cnn = X_test_tensor.unsqueeze(1)

class WineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Linear(16, len(np.unique(y)))

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


torch.manual_seed(42)

cnn_model = WineCNN()
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
cnn_epochs = 200

for epoch in range(cnn_epochs):
    cnn_model.train()
    cnn_logits = cnn_model(X_train_cnn)
    cnn_loss = cnn_criterion(cnn_logits, y_train_tensor)

    cnn_optimizer.zero_grad()
    cnn_loss.backward()
    cnn_optimizer.step()

cnn_model.eval()
with torch.no_grad():
    cnn_test_logits = cnn_model(X_test_cnn)
    cnn_predictions = torch.argmax(cnn_test_logits, dim=1)
    cnn_accuracy = (cnn_predictions == y_test_tensor).float().mean().item()

print(f"CNN test accuracy: {cnn_accuracy * 100:.2f}%")
print("First 10 predictions:", cnn_predictions[:10].tolist())
print("First 10 actual labels:", y_test_tensor[:10].tolist())
print("Note: CNNs are usually less natural than dense networks for small tabular datasets like Wine.")

CNN test accuracy: 77.78%
First 10 predictions: [0, 1, 0, 0, 1, 0, 0, 1, 1, 1]
First 10 actual labels: [0, 2, 0, 1, 1, 0, 0, 1, 1, 2]
Note: CNNs are usually less natural than dense networks for small tabular datasets like Wine.


## 9b. What is ANN, and how does it compare with CNN?

**ANN** stands for **Artificial Neural Network**. 
In this notebook, the earlier model is a **feed-forward ANN** made of fully connected (`Linear`) layers.
ANN = Artificial Neural Network (broad term)
MLP = Multilayer Perceptron (a specific type of ANN)

So for your assignment: the earlier fully connected model is more precisely an MLP calling it an ANN is still correct, just less specific
**In short: all MLPs are ANNs, but not all ANNs are MLPs.**

### ANN in simple terms
- it takes the input features and passes them through one or more hidden layers
- each layer learns useful patterns using weights and activation functions such as `ReLU`
- the final layer outputs class scores for prediction

### ANN vs CNN for this dataset
- **ANN** is usually a better fit for **tabular data** like the Wine dataset
- **CNN** is more commonly used for data with spatial structure, such as **images**, audio signals, or sequences
- since the Wine dataset only has 13 chemical features, the ANN is generally more natural and often performs better

In [14]:
ann_accuracy = test_accuracy * 100
cnn_accuracy_percent = cnn_accuracy * 100

print(f"ANN test accuracy: {ann_accuracy:.2f}%")
print(f"CNN test accuracy: {cnn_accuracy_percent:.2f}%")

if ann_accuracy > cnn_accuracy_percent:
    print("\nFor this Wine dataset, the ANN performed better than the CNN.")
elif cnn_accuracy_percent > ann_accuracy:
    print("\nFor this Wine dataset, the CNN performed better than the ANN.")
else:
    print("\nThe ANN and CNN achieved the same accuracy on this run.")

print("This supports the idea that ANNs are often more suitable for small structured tabular datasets.")

ANN test accuracy: 97.22%
CNN test accuracy: 77.78%

For this Wine dataset, the ANN performed better than the CNN.
This supports the idea that ANNs are often more suitable for small structured tabular datasets.


## Conclusion

This notebook provides a complete attempt for the assignment:
- the Wine dataset is loaded from scikit-learn
- features are standardized
- a PyTorch neural network is trained for classification
- performance is evaluated on the test set
- multiple hyperparameter settings are compared
- For this tabular Wine dataset, the ANN performed better, which is the expected outcome in most cases.
